# 02_benchmark_ml20m_full_positive

- treat all ratings as positive interactions
- keep users with at least 5 interactions
- no rating threshold in the main benchmark track
- leave-one-out split
- stronger SASRec baseline
- same coding style and same function structure as the earlier notebook

## 0. Setup

In [1]:
import math
import time
import random
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

DATA_DIR = Path("../data/ml-20m")
PROC_DIR = Path("../data/processed_benchmark")
MODEL_DIR = Path("../models")
RESULT_DIR = Path("../reports/results")

PROC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN = 200
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 20
EARLY_STOPPING_PATIENCE = 4

device: cuda


## 1. Load raw MovieLens 20M data

In [3]:
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
genome_tags = pd.read_csv(DATA_DIR / "genome-tags.csv")
genome_scores = pd.read_csv(DATA_DIR / "genome-scores.csv")
links = pd.read_csv(DATA_DIR / "links.csv")

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s", errors="coerce")
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit="s", errors="coerce")

movies["year"] = movies["title"].str.extract(r"\((\d{4})\)").astype(float)
movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)
movies["genres_list"] = movies["genres"].fillna("(no genres listed)").str.split("|")

tags = tags.dropna(subset=["tag"]).copy()
tags["tag"] = tags["tag"].astype(str).str.strip()
tags = tags[tags["tag"] != ""].copy()

print("ratings shape:", ratings.shape)
print("movies shape:", movies.shape)
print("tags shape:", tags.shape)
print("genome_tags shape:", genome_tags.shape)
print("genome_scores shape:", genome_scores.shape)
print("links shape:", links.shape)

print("\nratings time range:", ratings["timestamp"].min(), "->", ratings["timestamp"].max())
print("tags time range   :", tags["timestamp"].min(), "->", tags["timestamp"].max())

ratings shape: (20000263, 4)
movies shape: (27278, 6)
tags shape: (465541, 4)
genome_tags shape: (1128, 2)
genome_scores shape: (11709768, 3)
links shape: (27278, 3)

ratings time range: 1995-01-09 11:46:44 -> 2015-03-31 06:40:02
tags time range   : 2005-12-24 13:00:10 -> 2015-03-31 03:09:12


## 2. Benchmark preprocessing

This notebook uses the benchmark track:
- all ratings are positive interactions
- keep the last interaction if duplicate user-movie pairs exist
- keep users with at least 5 interactions

In [4]:
pair_counts = ratings.groupby(["userId", "movieId"]).size()

print("total rating rows:", len(ratings))
print("unique user-movie pairs:", len(pair_counts))
print("pairs with repeats:", (pair_counts > 1).sum())
print("rows inside repeated pairs:", pair_counts[pair_counts > 1].sum())
print("max repeats for one pair:", pair_counts.max())

total rating rows: 20000263
unique user-movie pairs: 20000263
pairs with repeats: 0
rows inside repeated pairs: 0
max repeats for one pair: 1


In [5]:
ratings_last = (
    ratings
    .sort_values(["userId", "movieId", "timestamp"])
    .drop_duplicates(["userId", "movieId"], keep="last")
    .reset_index(drop=True)
)

print("rows after last-rating dedup:", len(ratings_last))
print("users:", ratings_last["userId"].nunique())
print("items:", ratings_last["movieId"].nunique())

rows after last-rating dedup: 20000263
users: 138493
items: 26744


In [6]:
def kcore_filter_users_only(df, user_min=5):
    df = df.copy()

    while True:
        n_before = len(df)

        good_users = df["userId"].value_counts()
        good_users = good_users[good_users >= user_min].index
        df = df[df["userId"].isin(good_users)].copy()

        if len(df) == n_before:
            break

    return df

interactions = kcore_filter_users_only(ratings_last, user_min=5)
interactions = interactions.sort_values(["userId", "timestamp"]).reset_index(drop=True)

print("final benchmark interactions")
print("rows:", len(interactions))
print("users:", interactions["userId"].nunique())
print("items:", interactions["movieId"].nunique())

user_counts = interactions.groupby("userId").size()
item_counts = interactions.groupby("movieId").size()

print("\nuser count quantiles")
print(user_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nitem count quantiles")
print(item_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

final benchmark interactions
rows: 20000263
users: 138493
items: 26744

user count quantiles
0.25      35.00
0.50      68.00
0.75     155.00
0.90     334.00
0.95     520.00
0.99    1113.08
dtype: float64

item count quantiles
0.25        3.00
0.50       18.00
0.75      205.00
0.90     1305.70
0.95     3612.95
0.99    14388.69
dtype: float64


## 3. User split and ID mapping

In [7]:
user_ids = np.sort(interactions["userId"].unique())
movie_ids = np.sort(interactions["movieId"].unique())

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {m: i + 1 for i, m in enumerate(movie_ids)}
idx2item = {i: m for m, i in item2idx.items()}

interactions["user_idx"] = interactions["userId"].map(user2idx)
interactions["item_idx"] = interactions["movieId"].map(item2idx)

print("num users:", len(user2idx))
print("num items:", len(item2idx))
print(interactions[["userId", "movieId", "user_idx", "item_idx", "rating", "timestamp"]].head(10).to_string(index=False))

num users: 138493
num items: 26744
 userId  movieId  user_idx  item_idx  rating           timestamp
      1      924         0       908     3.5 2004-09-10 03:06:38
      1      919         0       903     3.5 2004-09-10 03:07:01
      1     2683         0      2598     3.5 2004-09-10 03:07:30
      1     1584         0      1533     3.5 2004-09-10 03:07:36
      1     1079         0      1058     4.0 2004-09-10 03:07:45
      1      653         0       646     3.0 2004-09-10 03:08:11
      1     2959         0      2874     4.0 2004-09-10 03:08:18
      1      337         0       334     3.5 2004-09-10 03:08:29
      1     1304         0      1276     3.0 2004-09-10 03:08:40
      1     3996         0      3903     4.0 2004-09-10 03:08:47


In [8]:
user_seq = (
    interactions.sort_values(["user_idx", "timestamp"])
    .groupby("user_idx")["item_idx"]
    .apply(list)
)

split_rows = []

for user_idx, seq in user_seq.items():
    split_rows.append({
        "user_idx": user_idx,
        "train_seq": seq[:-2],
        "val_seq": seq[:-2],
        "val_target": seq[-2],
        "test_seq": seq[:-1],
        "test_target": seq[-1],
        "full_len": len(seq)
    })

splits = pd.DataFrame(split_rows)

print("number of users in splits:", len(splits))
print("train seq length quantiles")
print(splits["train_seq"].apply(len).quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))
print("\nsplit preview")
print(splits.head(5).to_string(index=False))

number of users in splits: 138493
train seq length quantiles
0.25      33.00
0.50      66.00
0.75     153.00
0.90     332.00
0.95     518.00
0.99    1111.08
Name: train_seq, dtype: float64

split preview
 user_idx                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                train_seq                                                                                                                                                                                                          

In [9]:
interactions.to_pickle(PROC_DIR / "interactions_full_positive_u5.pkl")
splits.to_pickle(PROC_DIR / "splits_full_positive_u5.pkl")

pd.DataFrame({
    "userId": list(user2idx.keys()),
    "user_idx": list(user2idx.values())
}).to_pickle(PROC_DIR / "user_map_full_positive_u5.pkl")

pd.DataFrame({
    "movieId": list(item2idx.keys()),
    "item_idx": list(item2idx.values())
}).to_pickle(PROC_DIR / "item_map_full_positive_u5.pkl")

print("saved files in:", PROC_DIR)
print(sorted([p.name for p in PROC_DIR.iterdir()]))

saved files in: ../data/processed_benchmark
['interactions_full_positive_u5.pkl', 'item_map_full_positive_u5.pkl', 'splits_full_positive_u5.pkl', 'user_map_full_positive_u5.pkl']


## 4. Data loaders

In [10]:
interactions = pd.read_pickle(PROC_DIR / "interactions_full_positive_u5.pkl")
splits = pd.read_pickle(PROC_DIR / "splits_full_positive_u5.pkl")
item_map = pd.read_pickle(PROC_DIR / "item_map_full_positive_u5.pkl")
user_map = pd.read_pickle(PROC_DIR / "user_map_full_positive_u5.pkl")

num_users = len(user_map)
num_items = len(item_map)

print("num_users:", num_users)
print("num_items:", num_items)
print("splits:", splits.shape)

num_users: 138493
num_items: 26744
splits: (138493, 7)


In [11]:
def pad_seq(seq, max_len=200):
    seq = seq[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    out[-len(seq):] = seq
    return out

class TrainDataset(Dataset):
    def __init__(self, splits_df, max_len=200):
        self.max_len = max_len
        self.seqs = splits_df["train_seq"].tolist()

        keep = []
        for seq in self.seqs:
            if len(seq) >= 3:
                keep.append(seq)

        self.data = keep

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]

        tokens = seq[:-1]
        targets = seq[1:]

        tokens = pad_seq(tokens, self.max_len)
        targets = pad_seq(targets, self.max_len)

        return {
            "seq": torch.tensor(tokens, dtype=torch.long),
            "target_seq": torch.tensor(targets, dtype=torch.long),
        }

class EvalDataset(Dataset):
    def __init__(self, splits_df, mode="val", max_len=200):
        self.max_len = max_len

        if mode == "val":
            self.seq_col = "val_seq"
            self.target_col = "val_target"
        else:
            self.seq_col = "test_seq"
            self.target_col = "test_target"

        self.seq = splits_df[self.seq_col].tolist()
        self.target = splits_df[self.target_col].tolist()

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return {
            "seq": torch.tensor(pad_seq(self.seq[idx], self.max_len), dtype=torch.long),
            "target": torch.tensor(self.target[idx], dtype=torch.long),
        }

train_ds = TrainDataset(splits, max_len=MAX_LEN)
val_ds = EvalDataset(splits, mode="val", max_len=MAX_LEN)
test_ds = EvalDataset(splits, mode="test", max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("train sequences:", len(train_ds))
print("val users:", len(val_ds))
print("test users:", len(test_ds))

train sequences: 138493
val users: 138493
test users: 138493


## 5. Base model

In [12]:
class SASRecBase(nn.Module):
    def __init__(self, num_items, max_len=200, d_model=256, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.d_model = d_model

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)

    def _make_pos_ids(self, seq):
        mask = (seq > 0).long()
        pos_ids = torch.cumsum(mask, dim=1) * mask
        pos_ids = pos_ids.clamp(max=self.max_len)
        return pos_ids

    def _causal_mask(self, L, device):
        return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)

    def forward_sequence(self, seq):
        pos_ids = self._make_pos_ids(seq)

        x = self.item_emb(seq) + self.pos_emb(pos_ids)
        x = self.emb_norm(x)
        x = self.emb_dropout(x)

        pad_mask = (seq == 0)
        causal_mask = self._causal_mask(seq.size(1), seq.device)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        return x

    def encode(self, seq):
        x = self.forward_sequence(seq)
        lengths = (seq > 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        h = x[torch.arange(seq.size(0), device=seq.device), last_idx]
        return h

    def predict_all(self, seq):
        h = self.encode(seq)
        logits = h @ self.item_emb.weight.T
        logits[:, 0] = -1e9
        return logits

    def predict_all_positions(self, seq):
        x = self.forward_sequence(seq)
        logits = x @ self.item_emb.weight.T
        logits[:, :, 0] = -1e9
        return logits

model = SASRecBase(
    num_items=num_items,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

print(model)

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


SASRecBase(
  (item_emb): Embedding(26745, 256, padding_idx=0)
  (pos_emb): Embedding(201, 256, padding_idx=0)
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (emb_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=Tru

## 6. Training loop

In [13]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target_seq = batch["target_seq"].to(device)

        optimizer.zero_grad()

        logits = model.predict_all_positions(seq)

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_seq.reshape(-1),
            ignore_index=0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        valid_tokens = (target_seq > 0).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_topk(model, loader, device, k=10):
    model.eval()

    hits = 0.0
    ndcg = 0.0
    mrr = 0.0
    total = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target = batch["target"].to(device)

        logits = model.predict_all(seq)

        for i in range(seq.size(0)):
            seen = seq[i].unique()
            seen = seen[seen > 0]
            logits[i, seen] = -1e9

        topk = torch.topk(logits, k=k, dim=1).indices

        for i in range(seq.size(0)):
            tgt = target[i].item()
            pred = topk[i].tolist()

            if tgt in pred:
                rank = pred.index(tgt) + 1
                hits += 1.0
                ndcg += 1.0 / math.log2(rank + 1)
                mrr += 1.0 / rank

        total += seq.size(0)

    hr = hits / total
    ndcg = ndcg / total
    mrr = mrr / total

    return {
        f"HR@{k}": hr,
        f"NDCG@{k}": ndcg,
        f"MRR@{k}": mrr,
        f"Recall@{k}": hr,
    }

## 7. Train base benchmark model

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_val_ndcg = -1
best_state = None
bad_epochs = 0

train_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate_topk(model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        **val_metrics
    }
    history.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg:
        best_val_ndcg = val_metrics["NDCG@10"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= EARLY_STOPPING_PATIENCE:
        print(f"early stopping at epoch {epoch}")
        break

total_train_sec = time.perf_counter() - train_start
print("\ntraining finished in", round(total_train_sec, 2), "seconds")

epoch 01 | loss 6.5841 | HR@10 0.0341 | NDCG@10 0.0173 | MRR@10 0.0123 | 156.7s
epoch 02 | loss 5.6535 | HR@10 0.0411 | NDCG@10 0.0214 | MRR@10 0.0155 | 157.8s
epoch 03 | loss 5.4657 | HR@10 0.0434 | NDCG@10 0.0230 | MRR@10 0.0168 | 158.6s
epoch 04 | loss 5.3728 | HR@10 0.0448 | NDCG@10 0.0237 | MRR@10 0.0174 | 159.2s
epoch 05 | loss 5.3125 | HR@10 0.0456 | NDCG@10 0.0242 | MRR@10 0.0177 | 159.0s
epoch 06 | loss 5.2682 | HR@10 0.0464 | NDCG@10 0.0247 | MRR@10 0.0182 | 158.2s
epoch 07 | loss 5.2340 | HR@10 0.0472 | NDCG@10 0.0250 | MRR@10 0.0183 | 158.9s
epoch 08 | loss 5.2071 | HR@10 0.0468 | NDCG@10 0.0248 | MRR@10 0.0182 | 160.0s
epoch 09 | loss 5.1842 | HR@10 0.0469 | NDCG@10 0.0250 | MRR@10 0.0184 | 159.2s
epoch 10 | loss 5.1656 | HR@10 0.0480 | NDCG@10 0.0257 | MRR@10 0.0189 | 159.1s
epoch 11 | loss 5.1496 | HR@10 0.0487 | NDCG@10 0.0260 | MRR@10 0.0191 | 158.7s
epoch 12 | loss 5.1357 | HR@10 0.0483 | NDCG@10 0.0260 | MRR@10 0.0193 | 158.2s
epoch 13 | loss 5.1235 | HR@10 0.0491 | 

## 8. Evaluate and save outputs

In [15]:
model.load_state_dict(best_state)

test_start = time.perf_counter()
test_metrics = evaluate_topk(model, test_loader, device, k=10)
test_sec = time.perf_counter() - test_start

print("best validation NDCG@10:", round(best_val_ndcg, 6))
print("test metrics:")
for k, v in test_metrics.items():
    print(k, ":", round(v, 6))
print("test eval seconds:", round(test_sec, 2))

history_df = pd.DataFrame(history)
history_df

best validation NDCG@10: 0.026587
test metrics:
HR@10 : 0.043829
NDCG@10 : 0.023745
MRR@10 : 0.01767
Recall@10 : 0.043829
test eval seconds: 36.9


,epoch,train_loss,epoch_sec,HR@10,NDCG@10,MRR@10,Recall@10
0,1,6.584058,156.736311,0.034088,0.017332,0.012291,0.034088
1,2,5.653482,157.805712,0.041121,0.021427,0.015491,0.041121
2,3,5.465739,158.636453,0.043417,0.022952,0.016774,0.043417
3,4,5.372761,159.187303,0.044833,0.023749,0.017388,0.044833
4,5,5.312535,159.023297,0.045576,0.024178,0.017724,0.045576
5,6,5.268222,158.190871,0.046421,0.024721,0.018173,0.046421
6,7,5.234040,158.894954,0.047201,0.025002,0.018306,0.047201
7,8,5.207133,159.989982,0.046840,0.024839,0.018196,0.046840
8,9,5.184182,159.225651,0.046898,0.025019,0.018406,0.046898
9,10,5.165642,159.141590,0.047995,0.025657,0.018907,0.047995


In [16]:
torch.save(best_state, MODEL_DIR / "sasrec_base_full_positive_u5.pt")
history_df.to_csv(RESULT_DIR / "sasrec_base_full_positive_u5_history.csv", index=False)

pd.DataFrame([{
    "model": "sasrec_base_full_positive_u5",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "epochs_planned": EPOCHS,
    "train_total_sec": total_train_sec,
    "test_eval_sec": test_sec,
    **test_metrics
}]).to_csv(RESULT_DIR / "sasrec_base_full_positive_u5_test_metrics.csv", index=False)

print("saved model:", MODEL_DIR / "sasrec_base_full_positive_u5.pt")
print("saved history:", RESULT_DIR / "sasrec_base_full_positive_u5_history.csv")
print("saved test metrics:", RESULT_DIR / "sasrec_base_full_positive_u5_test_metrics.csv")

saved model: ../models/sasrec_base_full_positive_u5.pt
saved history: ../reports/results/sasrec_base_full_positive_u5_history.csv
saved test metrics: ../reports/results/sasrec_base_full_positive_u5_test_metrics.csv


## 9.  comparison
# Stronger base benchmar + Hybrid gated

In [17]:

#L — stronger benchmark config
# stronger benchmark config
MAX_LEN = 200
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 20
GRAD_CLIP = 1.0

print("MAX_LEN:", MAX_LEN)
print("D_MODEL:", D_MODEL)
print("N_HEADS:", N_HEADS)
print("N_LAYERS:", N_LAYERS)
print("DROPOUT:", DROPOUT)
print("BATCH_SIZE:", BATCH_SIZE)
print("LR:", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("EPOCHS:", EPOCHS)

MAX_LEN: 200
D_MODEL: 256
N_HEADS: 4
N_LAYERS: 4
DROPOUT: 0.1
BATCH_SIZE: 256
LR: 0.001
WEIGHT_DECAY: 1e-05
EPOCHS: 20


In [18]:

# M — rebuild loaders with the stronger config
train_ds = TrainDataset(splits, max_len=MAX_LEN)
val_ds = EvalDataset(splits, mode="val", max_len=MAX_LEN)
test_ds = EvalDataset(splits, mode="test", max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("train sequences:", len(train_ds))
print("val users:", len(val_ds))
print("test users:", len(test_ds))

train sequences: 138493
val users: 138493
test users: 138493


In [19]:

# N — stronger base model
base_model = SASRecBase(
    num_items=num_items,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

print(base_model)

SASRecBase(
  (item_emb): Embedding(26745, 256, padding_idx=0)
  (pos_emb): Embedding(201, 256, padding_idx=0)
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (emb_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=Tru

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [20]:

# O — optimizer and scheduler helpers
def build_optimizer(model, lr=1e-3, weight_decay=1e-5):
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

def build_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

def train_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target_seq = batch["target_seq"].to(device)

        optimizer.zero_grad()

        logits = model.predict_all_positions(seq)

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_seq.reshape(-1),
            ignore_index=0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        valid_tokens = (target_seq > 0).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_topk(model, loader, device, k=10):
    model.eval()

    hits = 0.0
    ndcg = 0.0
    mrr = 0.0
    total = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target = batch["target"].to(device)

        logits = model.predict_all(seq)

        for i in range(seq.size(0)):
            seen = seq[i].unique()
            seen = seen[seen > 0]
            logits[i, seen] = -1e9

        topk = torch.topk(logits, k=k, dim=1).indices

        for i in range(seq.size(0)):
            tgt = target[i].item()
            pred = topk[i].tolist()

            if tgt in pred:
                rank = pred.index(tgt) + 1
                hits += 1.0
                ndcg += 1.0 / math.log2(rank + 1)
                mrr += 1.0 / rank

        total += seq.size(0)

    hr = hits / total
    ndcg = ndcg / total
    mrr = mrr / total

    return {
        f"HR@{k}": hr,
        f"NDCG@{k}": ndcg,
        f"MRR@{k}": mrr,
        f"Recall@{k}": hr,
    }

In [21]:

# P — train the stronger benchmark base model
optimizer = build_optimizer(base_model, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = build_scheduler(optimizer, epochs=EPOCHS)

history_base = []
best_val_ndcg_base = -1
best_state_base = None

train_start_base = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(base_model, train_loader, optimizer, device, grad_clip=GRAD_CLIP)
    val_metrics = evaluate_topk(base_model, val_loader, device, k=10)

    scheduler.step()

    epoch_time = time.perf_counter() - t0
    lr_now = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        "lr": lr_now,
        **val_metrics
    }
    history_base.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"lr {lr_now:.6f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg_base:
        best_val_ndcg_base = val_metrics["NDCG@10"]
        best_state_base = {k: v.cpu().clone() for k, v in base_model.state_dict().items()}

total_train_sec_base = time.perf_counter() - train_start_base
print("\nbase benchmark training finished in", round(total_train_sec_base, 2), "seconds")

epoch 01 | loss 6.5879 | HR@10 0.0333 | NDCG@10 0.0169 | MRR@10 0.0119 | lr 0.000994 | 157.4s
epoch 02 | loss 5.6517 | HR@10 0.0407 | NDCG@10 0.0213 | MRR@10 0.0154 | lr 0.000976 | 157.6s
epoch 03 | loss 5.4652 | HR@10 0.0430 | NDCG@10 0.0226 | MRR@10 0.0164 | lr 0.000946 | 157.5s
epoch 04 | loss 5.3705 | HR@10 0.0449 | NDCG@10 0.0237 | MRR@10 0.0173 | lr 0.000905 | 157.6s
epoch 05 | loss 5.3074 | HR@10 0.0460 | NDCG@10 0.0242 | MRR@10 0.0177 | lr 0.000854 | 161.0s
epoch 06 | loss 5.2591 | HR@10 0.0471 | NDCG@10 0.0248 | MRR@10 0.0181 | lr 0.000794 | 160.3s
epoch 07 | loss 5.2202 | HR@10 0.0476 | NDCG@10 0.0254 | MRR@10 0.0188 | lr 0.000727 | 160.7s
epoch 08 | loss 5.1868 | HR@10 0.0481 | NDCG@10 0.0259 | MRR@10 0.0191 | lr 0.000655 | 164.6s
epoch 09 | loss 5.1576 | HR@10 0.0489 | NDCG@10 0.0261 | MRR@10 0.0192 | lr 0.000578 | 160.0s
epoch 10 | loss 5.1305 | HR@10 0.0489 | NDCG@10 0.0265 | MRR@10 0.0197 | lr 0.000500 | 158.9s
epoch 11 | loss 5.1056 | HR@10 0.0495 | NDCG@10 0.0267 | MRR

In [22]:

# Q — evaluate and save stronger benchmark base
base_model.load_state_dict(best_state_base)

test_start_base = time.perf_counter()
test_metrics_base = evaluate_topk(base_model, test_loader, device, k=10)
test_sec_base = time.perf_counter() - test_start_base

print("best validation NDCG@10:", round(best_val_ndcg_base, 6))
print("base benchmark test metrics:")
for k, v in test_metrics_base.items():
    print(k, ":", round(v, 6))
print("base benchmark test eval seconds:", round(test_sec_base, 2))

history_base_df = pd.DataFrame(history_base)
history_base_df

best validation NDCG@10: 0.027676
base benchmark test metrics:
HR@10 : 0.044385
NDCG@10 : 0.023999
MRR@10 : 0.017827
Recall@10 : 0.044385
base benchmark test eval seconds: 37.5


,epoch,train_loss,epoch_sec,lr,HR@10,NDCG@10,MRR@10,Recall@10
0,1,6.587901,157.421129,0.000994,0.033309,0.016863,0.011922,0.033309
1,2,5.651681,157.628525,0.000976,0.040674,0.021272,0.015430,0.040674
2,3,5.465193,157.475675,0.000946,0.043020,0.022602,0.016443,0.043020
3,4,5.370525,157.580940,0.000905,0.044948,0.023673,0.017258,0.044948
4,5,5.307396,160.991739,0.000854,0.045959,0.024239,0.017687,0.045959
5,6,5.259096,160.341232,0.000794,0.047071,0.024785,0.018058,0.047071
6,7,5.220230,160.678703,0.000727,0.047576,0.025447,0.018775,0.047576
7,8,5.186762,164.628133,0.000655,0.048140,0.025870,0.019133,0.048140
8,9,5.157600,160.024183,0.000578,0.048854,0.026089,0.019202,0.048854
9,10,5.130534,158.851625,0.000500,0.048948,0.026475,0.019681,0.048948


In [23]:

# R — save stronger benchmark base outputs
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = Path("../reports/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(best_state_base, MODEL_DIR / "benchmark_sasrec_base.pt")
history_base_df.to_csv(RESULT_DIR / "benchmark_sasrec_base_history.csv", index=False)

pd.DataFrame([{
    "model": "benchmark_sasrec_base",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS,
    "train_total_sec": total_train_sec_base,
    "test_eval_sec": test_sec_base,
    **test_metrics_base
}]).to_csv(RESULT_DIR / "benchmark_sasrec_base_test_metrics.csv", index=False)

print("saved model:", MODEL_DIR / "benchmark_sasrec_base.pt")
print("saved history:", RESULT_DIR / "benchmark_sasrec_base_history.csv")
print("saved test metrics:", RESULT_DIR / "benchmark_sasrec_base_test_metrics.csv")

saved model: ../models/benchmark_sasrec_base.pt
saved history: ../reports/results/benchmark_sasrec_base_history.csv
saved test metrics: ../reports/results/benchmark_sasrec_base_test_metrics.csv


In [46]:

# S — build structured feature tensors for the full-positive benchmark
# # S0 — build item_meta for the benchmark notebook
# S0 — build item_meta before S

import pandas as pd
import numpy as np
from pathlib import Path

if "item_map" in globals():
    if isinstance(item_map, pd.DataFrame):
        item_map_df = item_map.copy()
    else:
        item_map_df = pd.DataFrame(item_map)
elif "interactions" in globals() and {"movieId", "item_idx"}.issubset(interactions.columns):
    item_map_df = interactions[["movieId", "item_idx"]].drop_duplicates().sort_values("item_idx").reset_index(drop=True)
elif "item2idx" in globals():
    item_map_df = pd.DataFrame({
        "movieId": list(item2idx.keys()),
        "item_idx": list(item2idx.values())
    }).sort_values("item_idx").reset_index(drop=True)
else:
    raise ValueError("Need item_map, or interactions with movieId/item_idx, or item2idx")

movies = movies.copy()
movies["year"] = pd.to_numeric(
    movies["title"].astype(str).str.extract(r"\((\d{4})\)")[0],
    errors="coerce"
)
movies["clean_title"] = movies["title"].astype(str).str.replace(r"\s*\(\d{4}\)$", "", regex=True)
movies["genres_text"] = movies["genres"].fillna("").str.replace("|", " ", regex=False)

tags = tags.copy()
tags = tags.dropna(subset=["tag"]).copy()
tags["tag_lower"] = tags["tag"].astype(str).str.strip().str.lower()
tags = tags[tags["tag_lower"] != ""].copy()

tag_counts = (
    tags.groupby(["movieId", "tag_lower"])
        .size()
        .reset_index(name="count")
        .sort_values(["movieId", "count", "tag_lower"], ascending=[True, False, True])
)

top_tags = tag_counts.groupby("movieId").head(5).copy()

top_tags_text = (
    top_tags.groupby("movieId")["tag_lower"]
    .apply(lambda s: " | ".join(s.tolist()))
    .reset_index(name="top_tags_text")
)

item_meta = item_map_df.merge(
    movies[["movieId", "title", "clean_title", "genres", "genres_text", "year"]],
    on="movieId",
    how="left"
).merge(
    top_tags_text,
    on="movieId",
    how="left"
)

item_meta["top_tags_text"] = item_meta["top_tags_text"].fillna("")
item_meta["movie_text"] = (
    item_meta["clean_title"].fillna("") + ". " +
    item_meta["genres_text"].fillna("") + ". " +
    item_meta["top_tags_text"].fillna("")
).str.strip()

item_meta = item_meta.sort_values("item_idx").reset_index(drop=True)

print("item_meta shape:", item_meta.shape)
print(
    item_meta[["item_idx", "movieId", "clean_title", "genres", "top_tags_text"]]
    .head(10)
    .to_string(index=False)
)

item_meta shape: (26744, 9)
 item_idx  movieId                 clean_title                                      genres                                                                         top_tags_text
        1        1                   Toy Story Adventure|Animation|Children|Comedy|Fantasy                           pixar | animation | disney | tom hanks | computer animation
        2        2                     Jumanji                  Adventure|Children|Fantasy                         robin williams | fantasy | time travel | animals | board game
        3        3            Grumpier Old Men                              Comedy|Romance                        moldy | old | sequel | clv | comedinha de velhinhos engraã§ada
        4        4           Waiting to Exhale                        Comedy|Drama|Romance                                              characters | chick flick | clv | revenge
        5        5 Father of the Bride Part II                                      Com

In [47]:


# S — build structured feature tensors

genre_dummies = item_meta["genres"].fillna("").str.get_dummies(sep="|")
genre_cols = genre_dummies.columns.tolist()

year_vocab = [
    "unknown",
    "<=1950",
    "1951-1960",
    "1961-1970",
    "1971-1980",
    "1981-1990",
    "1991-2000",
    "2001-2010",
    "2011-2020",
]
year2idx = {y: i for i, y in enumerate(year_vocab)}

item_meta["year_bucket"] = pd.cut(
    item_meta["year"],
    bins=[0, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020],
    labels=year_vocab[1:],
    include_lowest=True
).astype("object").fillna("unknown")

genre_mat = np.zeros((num_items + 1, len(genre_cols)), dtype=np.float32)
year_ids = np.zeros(num_items + 1, dtype=np.int64)

idx = item_meta["item_idx"].to_numpy()
genre_mat[idx] = genre_dummies.to_numpy(dtype=np.float32)
year_ids[idx] = item_meta["year_bucket"].map(year2idx).fillna(0).astype(int).to_numpy()

genre_tensor = torch.tensor(genre_mat, dtype=torch.float32)
year_tensor = torch.tensor(year_ids, dtype=torch.long)

print("genre tensor:", genre_tensor.shape)
print("year tensor:", year_tensor.shape)
print("genre cols:", len(genre_cols))

genre tensor: torch.Size([26745, 20])
year tensor: torch.Size([26745])
genre cols: 20


In [48]:


# T — load or build text embeddings

import re
import unicodedata
from pathlib import Path
from sentence_transformers import SentenceTransformer

TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_BATCH_SIZE = 256
TEXT_DIR = Path("../data/processed")
TEXT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_PATH = TEXT_DIR / "benchmark_text_minilm_tensor.pt"

def clean_text_basic(x):
    x = "" if pd.isna(x) else str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("|", " ")
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x

if TEXT_PATH.exists():
    text_tensor = torch.load(TEXT_PATH, map_location="cpu")
    print("loaded existing text tensor:", TEXT_PATH)
    print("text tensor shape:", text_tensor.shape)
else:
    item_text_df = item_meta[["item_idx", "movieId", "clean_title", "genres_text", "top_tags_text"]].copy()

    item_text_df["movie_text_clean"] = (
        item_text_df["clean_title"].fillna("").map(clean_text_basic) + ". " +
        item_text_df["genres_text"].fillna("").map(clean_text_basic) + ". " +
        item_text_df["top_tags_text"].fillna("").map(clean_text_basic)
    ).str.strip()

    text_model = SentenceTransformer(TEXT_MODEL_NAME)

    texts = item_text_df.sort_values("item_idx")["movie_text_clean"].tolist()

    text_emb = text_model.encode(
        texts,
        batch_size=TEXT_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    text_emb_full = np.zeros((num_items + 1, text_emb.shape[1]), dtype=np.float32)
    text_emb_full[1:] = text_emb.astype(np.float32)

    text_tensor = torch.tensor(text_emb_full, dtype=torch.float32)
    torch.save(text_tensor, TEXT_PATH)

    print("built and saved text tensor:", TEXT_PATH)
    print("text tensor shape:", text_tensor.shape)

Batches: 100%|██████████| 105/105 [00:02<00:00, 51.81it/s]


built and saved text tensor: ../data/processed/benchmark_text_minilm_tensor.pt
text tensor shape: torch.Size([26745, 384])


In [49]:

# U — hybrid gated model
class SASRecHybridGated(nn.Module):
    def __init__(
        self,
        num_items,
        genre_tensor,
        year_tensor,
        text_tensor,
        max_len=200,
        d_model=256,
        n_heads=4,
        n_layers=4,
        dropout=0.1,
    ):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.d_model = d_model

        self.register_buffer("genre_features", genre_tensor)
        self.register_buffer("year_ids", year_tensor)
        self.register_buffer("text_features", text_tensor)

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        self.genre_proj = nn.Linear(self.genre_features.shape[1], d_model)
        self.year_emb = nn.Embedding(int(self.year_ids.max().item()) + 1, d_model)
        self.text_proj = nn.Linear(self.text_features.shape[1], d_model)

        self.struct_norm = nn.LayerNorm(d_model)
        self.text_norm = nn.LayerNorm(d_model)

        self.struct_dropout = nn.Dropout(dropout)
        self.text_dropout = nn.Dropout(dropout)

        self.struct_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

        self.text_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.normal_(self.year_emb.weight, std=0.02)

        nn.init.xavier_uniform_(self.genre_proj.weight)
        nn.init.zeros_(self.genre_proj.bias)

        nn.init.xavier_uniform_(self.text_proj.weight)
        nn.init.zeros_(self.text_proj.bias)

        nn.init.xavier_uniform_(self.struct_gate[0].weight)
        nn.init.zeros_(self.struct_gate[0].bias)
        nn.init.xavier_uniform_(self.struct_gate[2].weight)
        nn.init.zeros_(self.struct_gate[2].bias)

        nn.init.xavier_uniform_(self.text_gate[0].weight)
        nn.init.zeros_(self.text_gate[0].bias)
        nn.init.xavier_uniform_(self.text_gate[2].weight)
        nn.init.zeros_(self.text_gate[2].bias)

    def _make_pos_ids(self, seq):
        mask = (seq > 0).long()
        pos_ids = torch.cumsum(mask, dim=1) * mask
        pos_ids = pos_ids.clamp(max=self.max_len)
        return pos_ids

    def _causal_mask(self, L, device):
        return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)

    def item_token_embed(self, seq):
        item_vec = self.item_emb(seq)

        struct_vec = (
            self.genre_proj(self.genre_features[seq]) +
            self.year_emb(self.year_ids[seq])
        )
        struct_vec = self.struct_norm(struct_vec)
        struct_vec = self.struct_dropout(struct_vec)

        text_vec = self.text_proj(self.text_features[seq])
        text_vec = self.text_norm(text_vec)
        text_vec = self.text_dropout(text_vec)

        g_struct = self.struct_gate(torch.cat([item_vec, struct_vec], dim=-1))
        g_text = self.text_gate(torch.cat([item_vec, text_vec], dim=-1))

        x = item_vec + g_struct * struct_vec + g_text * text_vec
        return x

    def forward_sequence(self, seq):
        pos_ids = self._make_pos_ids(seq)

        x = self.item_token_embed(seq) + self.pos_emb(pos_ids)
        x = self.emb_norm(x)
        x = self.emb_dropout(x)

        pad_mask = (seq == 0)
        causal_mask = self._causal_mask(seq.size(1), seq.device)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        return x

    def encode(self, seq):
        x = self.forward_sequence(seq)
        lengths = (seq > 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        h = x[torch.arange(seq.size(0), device=seq.device), last_idx]
        return h

    def predict_all(self, seq):
        h = self.encode(seq)
        logits = h @ self.item_emb.weight.T
        logits[:, 0] = -1e9
        return logits

    def predict_all_positions(self, seq):
        x = self.forward_sequence(seq)
        logits = x @ self.item_emb.weight.T
        logits[:, :, 0] = -1e9
        return logits

In [50]:

# V — warm-start hybrid from the best base model
def load_submodule_from_flat_state(submodule, state_dict, prefix):
    sub_state = {}
    for k, v in state_dict.items():
        if k.startswith(prefix):
            sub_state[k[len(prefix):]] = v
    submodule.load_state_dict(sub_state)

hybrid_model = SASRecHybridGated(
    num_items=num_items,
    genre_tensor=genre_tensor,
    year_tensor=year_tensor,
    text_tensor=text_tensor,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
).to(device)

base_state = torch.load(MODEL_DIR / "benchmark_sasrec_base.pt", map_location="cpu", weights_only=True)

hybrid_model.item_emb.load_state_dict({
    "weight": base_state["item_emb.weight"]
})
hybrid_model.pos_emb.load_state_dict({
    "weight": base_state["pos_emb.weight"]
})
hybrid_model.emb_norm.load_state_dict({
    "weight": base_state["emb_norm.weight"],
    "bias": base_state["emb_norm.bias"],
})
hybrid_model.final_norm.load_state_dict({
    "weight": base_state["final_norm.weight"],
    "bias": base_state["final_norm.bias"],
})

for i in range(N_LAYERS):
    load_submodule_from_flat_state(
        hybrid_model.encoder.layers[i],
        base_state,
        prefix=f"encoder.layers.{i}."
    )

print(hybrid_model)

SASRecHybridGated(
  (item_emb): Embedding(26745, 256, padding_idx=0)
  (pos_emb): Embedding(201, 256, padding_idx=0)
  (genre_proj): Linear(in_features=20, out_features=256, bias=True)
  (year_emb): Embedding(9, 256)
  (text_proj): Linear(in_features=384, out_features=256, bias=True)
  (struct_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (text_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (struct_dropout): Dropout(p=0.1, inplace=False)
  (text_dropout): Dropout(p=0.1, inplace=False)
  (struct_gate): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): Sigmoid()
  )
  (text_gate): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): Sigmoid()
  )
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (emb_norm

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [ ]:

# W — stage 1 fine-tune heads only
for p in hybrid_model.item_emb.parameters():
    p.requires_grad = False
for p in hybrid_model.pos_emb.parameters():
    p.requires_grad = False
for p in hybrid_model.encoder.parameters():
    p.requires_grad = False
for p in hybrid_model.emb_norm.parameters():
    p.requires_grad = False
for p in hybrid_model.final_norm.parameters():
    p.requires_grad = False

LR_STAGE1 = 5e-4
EPOCHS_STAGE1 = 2

optimizer = build_optimizer(hybrid_model, lr=LR_STAGE1, weight_decay=WEIGHT_DECAY)

history_hybrid_stage1 = []
best_val_ndcg_hybrid = -1
best_state_hybrid = None

for epoch in range(1, EPOCHS_STAGE1 + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(hybrid_model, train_loader, optimizer, device, grad_clip=GRAD_CLIP)
    val_metrics = evaluate_topk(hybrid_model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0

    row = {
        "stage": 1,
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        **val_metrics
    }
    history_hybrid_stage1.append(row)

    print(
        f"stage1 epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg_hybrid:
        best_val_ndcg_hybrid = val_metrics["NDCG@10"]
        best_state_hybrid = {k: v.cpu().clone() for k, v in hybrid_model.state_dict().items()}

In [ ]:

# X — stage 2 unfreeze all and continue fine-tuning
for p in hybrid_model.parameters():
    p.requires_grad = True

hybrid_model.load_state_dict(best_state_hybrid)

LR_STAGE2 = 3e-4
EPOCHS_STAGE2 = 6

optimizer = build_optimizer(hybrid_model, lr=LR_STAGE2, weight_decay=WEIGHT_DECAY)
scheduler = build_scheduler(optimizer, epochs=EPOCHS_STAGE2)

history_hybrid_stage2 = []
train_start_hybrid = time.perf_counter()

for epoch in range(1, EPOCHS_STAGE2 + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(hybrid_model, train_loader, optimizer, device, grad_clip=GRAD_CLIP)
    val_metrics = evaluate_topk(hybrid_model, val_loader, device, k=10)

    scheduler.step()

    epoch_time = time.perf_counter() - t0
    lr_now = optimizer.param_groups[0]["lr"]

    row = {
        "stage": 2,
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        "lr": lr_now,
        **val_metrics
    }
    history_hybrid_stage2.append(row)

    print(
        f"stage2 epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"lr {lr_now:.6f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg_hybrid:
        best_val_ndcg_hybrid = val_metrics["NDCG@10"]
        best_state_hybrid = {k: v.cpu().clone() for k, v in hybrid_model.state_dict().items()}

total_train_sec_hybrid = time.perf_counter() - train_start_hybrid
print("\nhybrid fine-tuning finished in", round(total_train_sec_hybrid, 2), "seconds")

In [ ]:

# X — stage 2 unfreeze all and continue fine-tuning
for p in hybrid_model.parameters():
    p.requires_grad = True

hybrid_model.load_state_dict(best_state_hybrid)

LR_STAGE2 = 3e-4
EPOCHS_STAGE2 = 6

optimizer = build_optimizer(hybrid_model, lr=LR_STAGE2, weight_decay=WEIGHT_DECAY)
scheduler = build_scheduler(optimizer, epochs=EPOCHS_STAGE2)

history_hybrid_stage2 = []
train_start_hybrid = time.perf_counter()

for epoch in range(1, EPOCHS_STAGE2 + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(hybrid_model, train_loader, optimizer, device, grad_clip=GRAD_CLIP)
    val_metrics = evaluate_topk(hybrid_model, val_loader, device, k=10)

    scheduler.step()

    epoch_time = time.perf_counter() - t0
    lr_now = optimizer.param_groups[0]["lr"]

    row = {
        "stage": 2,
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        "lr": lr_now,
        **val_metrics
    }
    history_hybrid_stage2.append(row)

    print(
        f"stage2 epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"lr {lr_now:.6f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg_hybrid:
        best_val_ndcg_hybrid = val_metrics["NDCG@10"]
        best_state_hybrid = {k: v.cpu().clone() for k, v in hybrid_model.state_dict().items()}

total_train_sec_hybrid = time.perf_counter() - train_start_hybrid
print("\nhybrid fine-tuning finished in", round(total_train_sec_hybrid, 2), "seconds")